# X-means Klaszterezés - Bank Marketing Dataset
## Age és Balance alapján

### Célkitűzések:
- X-means klaszterezés automatikus klaszterszám meghatározással (kmin=2, kmax=10)
- 2 változó: **age** (életkor) és **balance** (egyenleg)
- BIC (Bayesian Information Criterion) alapú optimalizálás
- Validációs metrikák és összehasonlítás K-means-szel
- Klaszter profilozás és exportálás

## 1. Könyvtárak importálása

In [ ]:
# Alapvető könyvtárak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn könyvtárak
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Matplotlib beállítások
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Könyvtárak sikeresen betöltve!")

## 2. X-means Osztály Implementáció

In [ ]:
class XMeans:
    """
    X-Means klaszterezési algoritmus implementáció.
    Automatikusan meghatározza az optimális klaszterszámot BIC alapján.
    """
    
    def __init__(self, kmin=2, kmax=10, random_state=42):
        """
        Paraméterek:
        -----------
        kmin : int
            Minimális klaszterszám (default: 2)
        kmax : int
            Maximális klaszterszám (default: 10)
        random_state : int
            Véletlenszám generátor seed (default: 42)
        """
        self.kmin = kmin
        self.kmax = kmax
        self.random_state = random_state
        self.labels_ = None
        self.cluster_centers_ = None
        self.n_clusters_ = None
        self.bic_scores_ = []
        
    def _calculate_bic(self, X, labels, centers):
        """
        BIC (Bayesian Information Criterion) számítása.
        Alacsonyabb BIC érték = jobb modell.
        """
        n_samples, n_features = X.shape
        n_clusters = len(centers)
        
        # Log-likelihood számítása
        variance = 0
        for i in range(n_clusters):
            cluster_points = X[labels == i]
            if len(cluster_points) > 0:
                variance += np.sum((cluster_points - centers[i]) ** 2)
        
        variance = variance / (n_samples - n_clusters)
        
        if variance == 0:
            variance = 1e-10
            
        log_likelihood = -n_samples * np.log(2 * np.pi) / 2 - \
                        n_samples * n_features * np.log(variance) / 2 - \
                        (n_samples - n_clusters) / 2
        
        # BIC = log(n) * k - 2 * log_likelihood
        # k = paraméterek száma (középpontok + klaszter méretek)
        n_parameters = n_clusters * n_features + n_clusters
        bic = n_parameters * np.log(n_samples) - 2 * log_likelihood
        
        return bic
    
    def fit(self, X):
        """
        X-Means illesztése az adatokra.
        Minden k értéket tesztel kmin-től kmax-ig, és kiválasztja a legjobb BIC-t.
        """
        best_bic = np.inf
        best_k = self.kmin
        
        print("="*70)
        print("X-MEANS KLASZTEREZÉS FUTTATÁSA")
        print(f"Paraméterek: kmin={self.kmin}, kmax={self.kmax}, random_state={self.random_state}")
        print("="*70)
        
        # Próbáljuk ki az összes k értéket kmin-től kmax-ig
        for k in range(self.kmin, self.kmax + 1):
            # K-means++ inicializálás
            kmeans = KMeans(n_clusters=k, init='k-means++', random_state=self.random_state, n_init=50)
            labels = kmeans.fit_predict(X)
            centers = kmeans.cluster_centers_
            
            bic = self._calculate_bic(X, labels, centers)
            self.bic_scores_.append({'k': k, 'bic': bic})
            
            print(f"k = {k:2d}: BIC = {bic:>12,.2f}")
            
            # Keressük a legkisebb BIC értéket (alacsonyabb = jobb)
            if bic < best_bic:
                best_bic = bic
                best_k = k
                self.labels_ = labels
                self.cluster_centers_ = centers
        
        self.n_clusters_ = best_k
        print("="*70)
        print(f"✓ Optimális klaszterszám: k = {self.n_clusters_}")
        print(f"✓ Legjobb BIC érték: {best_bic:,.2f}")
        print("="*70)
        
        return self
    
    def predict(self, X):
        """
        Új adatpontok klaszterekhez rendelése.
        """
        distances = np.sqrt(((X[:, np.newaxis] - self.cluster_centers_) ** 2).sum(axis=2))
        return np.argmin(distances, axis=1)

print("✓ X-means osztály definiálva!")

## 3. Adatok betöltése és előkészítése

In [ ]:
# Adathalmaz beolvasása
root = r"bank+marketing\bank\bank-full.csv"
df = pd.read_csv(root, sep=';')

print("="*70)
print("ADATOK BETÖLTÉSE")
print("="*70)
print(f"Sorok száma: {len(df):,}")
print(f"Oszlopok száma: {len(df.columns)}")
print("\nElső 5 sor:")
print(df[['age', 'balance']].head())
print("\nStatisztikák:")
print(df[['age', 'balance']].describe())

In [ ]:
# Feature Selection: age és balance
features = ['age', 'balance']
X = df[features].copy()

print("\n" + "="*70)
print("FEATURE SELECTION ÉS NORMALIZÁLÁS")
print("="*70)
print(f"Kiválasztott változók: {features}")
print(f"Adatpontok száma: {len(X):,}")

# Normalizálás: StandardScaler
scaler = StandardScaler()
data_scaled = scaler.fit_transform(X)

print("\n✓ Adatok normalizálva (StandardScaler)!")
print(f"  data_scaled alakja: {data_scaled.shape}")
print(f"  Átlag: {data_scaled.mean(axis=0)}")
print(f"  Szórás: {data_scaled.std(axis=0)}")

## 4. X-means Klaszterezés Futtatása

In [ ]:
# X-means klaszterezés
xmeans = XMeans(kmin=2, kmax=10, random_state=42)
xmeans.fit(data_scaled)

# Címkék hozzáadása az eredeti adatokhoz
df['cluster'] = xmeans.labels_

# Középpontok visszaalakítása az eredeti skálára
centroids_original = scaler.inverse_transform(xmeans.cluster_centers_)

print("\n✓ Klaszter címkék hozzáadva az adathalmazhoz!")

## 5. BIC Értékek Vizualizációja

In [ ]:
# BIC értékek DataFrame-be
bic_df = pd.DataFrame(xmeans.bic_scores_)

# BIC plot
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(bic_df['k'], bic_df['bic'], 'bo-', linewidth=2.5, markersize=10, label='BIC értékek')

# Optimális k megjelölése
optimal_idx = bic_df['bic'].idxmin()
optimal_k = bic_df.loc[optimal_idx, 'k']
optimal_bic = bic_df.loc[optimal_idx, 'bic']

ax.scatter([optimal_k], [optimal_bic], color='red', s=400, zorder=5, 
          marker='*', edgecolors='darkred', linewidths=2, label=f'Optimális k={int(optimal_k)}')

ax.axvline(x=optimal_k, color='red', linestyle='--', alpha=0.5, linewidth=2)

# Értékek annotálása
for idx, row in bic_df.iterrows():
    ax.annotate(f'{row["bic"]:,.0f}', 
               (row['k'], row['bic']), 
               textcoords="offset points", 
               xytext=(0,10), 
               ha='center', 
               fontsize=9,
               bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))

ax.set_xlabel('Klaszterek száma (k)', fontsize=12, fontweight='bold')
ax.set_ylabel('BIC (Bayesian Information Criterion)', fontsize=12, fontweight='bold')
ax.set_title('X-means BIC értékek - Optimális klaszterszám kiválasztása\n' + 
            '(Alacsonyabb BIC = jobb modell)', 
            fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(bic_df['k'])
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.savefig('ábrák/xmeans_bic_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ BIC plot mentve: ábrák/xmeans_bic_plot.png")

## 6. Validációs Metrikák Számítása

In [ ]:
# Validációs metrikák számítása
silhouette = silhouette_score(data_scaled, xmeans.labels_)
davies_bouldin = davies_bouldin_score(data_scaled, xmeans.labels_)
calinski_harabasz = calinski_harabasz_score(data_scaled, xmeans.labels_)

# Metrikák DataFrame-be
metrics_df = pd.DataFrame({
    'Metrika': ['Optimális k', 'Silhouette Score', 'Davies-Bouldin Index', 'Calinski-Harabasz Index'],
    'Érték': [xmeans.n_clusters_, silhouette, davies_bouldin, calinski_harabasz],
    'Értelmezés': [
        'Klaszterek száma',
        'Magasabb = jobb (max: 1)',
        'Alacsonyabb = jobb',
        'Magasabb = jobb'
    ]
})

print("="*90)
print("VALIDÁCIÓS METRIKÁK - X-MEANS")
print("="*90)
print(metrics_df.to_string(index=False))
print("="*90)

# Exportálás CSV-be
metrics_df.to_csv('ábrák/metrics_xmeans.csv', index=False, encoding='utf-8-sig')
print("\n✓ Metrikák exportálva: ábrák/metrics_xmeans.csv")

## 7. Klaszter Profilozás

In [ ]:
# Klaszter profilozás
cluster_profile = []

for cluster_id in range(xmeans.n_clusters_):
    cluster_data = df[df['cluster'] == cluster_id]
    
    profile = {
        'Klaszter': cluster_id,
        'Átlag age': cluster_data['age'].mean(),
        'Szórás age': cluster_data['age'].std(),
        'Átlag balance': cluster_data['balance'].mean(),
        'Szórás balance': cluster_data['balance'].std(),
        'Elemszám (db)': len(cluster_data),
        'Elemszám (%)': len(cluster_data) / len(df) * 100
    }
    cluster_profile.append(profile)

profile_df = pd.DataFrame(cluster_profile)

print("\n" + "="*100)
print("KLASZTER PROFILOZÁS - X-MEANS")
print("="*100)
print(profile_df.to_string(index=False))
print("="*100)

# Exportálás CSV-be
profile_df.to_csv('ábrák/profil_xmeans.csv', index=False, encoding='utf-8-sig')
print("\n✓ Klaszter profilok exportálva: ábrák/profil_xmeans.csv")

## 8. Centroidok Exportálása

In [ ]:
# Centroidok DataFrame
centroids_df = pd.DataFrame(
    centroids_original,
    columns=['Átlag életkor (év)', 'Átlag egyenleg (€)']
)
centroids_df.index = [f'Cluster_{i}' for i in range(xmeans.n_clusters_)]

# Klaszter méret hozzáadása
cluster_sizes = df['cluster'].value_counts().sort_index()
centroids_df['Ügyfelek száma'] = cluster_sizes.values
centroids_df['Arány (%)'] = (centroids_df['Ügyfelek száma'] / len(df) * 100).round(2)

print("\n" + "="*90)
print("KLASZTER CENTROIDOK - X-MEANS (eredeti skálán)")
print("="*90)
print(centroids_df.to_string())
print("="*90)

# Exportálás CSV-be
centroids_df.to_csv('ábrák/centroids_xmeans.csv', encoding='utf-8-sig')
print("\n✓ Centroidok exportálva: ábrák/centroids_xmeans.csv")

## 9. Scatter Plot - Age vs Balance

In [ ]:
# Scatter plot - Age vs Balance színkódolt klaszterekkel
fig, ax = plt.subplots(figsize=(14, 10))

# Színek definiálása
colors = plt.cm.Set3(np.linspace(0, 1, xmeans.n_clusters_))

# Klaszterek ábrázolása
for cluster_id in range(xmeans.n_clusters_):
    cluster_data = df[df['cluster'] == cluster_id]
    ax.scatter(cluster_data['age'], cluster_data['balance'], 
              label=f'Cluster {cluster_id} ({len(cluster_data):,} ügyfél)', 
              alpha=0.6, s=30, c=[colors[cluster_id]], edgecolors='black', linewidths=0.3)

# Centroidok ábrázolása
ax.scatter(centroids_original[:, 0], centroids_original[:, 1], 
          marker='*', s=800, c='red', edgecolors='black', linewidths=2.5, 
          label='Centroidok', zorder=10)

# Centroid címkék
for i, (x, y) in enumerate(centroids_original):
    ax.annotate(f'C{i}\n({x:.0f}, {y:,.0f})', 
               (x, y), 
               xytext=(15, 15), 
               textcoords='offset points',
               fontsize=10,
               fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.8, edgecolor='black'),
               arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0', lw=1.5))

ax.set_xlabel('Életkor (év)', fontsize=13, fontweight='bold')
ax.set_ylabel('Számlaegyenleg (€)', fontsize=13, fontweight='bold')
ax.set_title(f'X-Means Klaszterezés - Age vs Balance (k={xmeans.n_clusters_})\n' + 
            f'Silhouette Score: {silhouette:.4f}', 
            fontsize=15, fontweight='bold', pad=20)
ax.legend(loc='upper right', fontsize=10, framealpha=0.95, edgecolor='black')
ax.grid(True, alpha=0.3, linestyle='--')

# Y tengely formázása
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'.replace(',', ' ')))

plt.tight_layout()
plt.savefig('ábrák/xmeans_scatter_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Scatter plot mentve: ábrák/xmeans_scatter_plot.png")

## 10. K-means Összehasonlítás
### Összehasonlítás a kmeans-v1.ipynb eredményeivel

In [ ]:
# K-means eredmények betöltése (feltételezve, hogy kmeans-v1.ipynb már futott)
# Ha még nem futott, akkor manuálisan adjuk meg az értékeket

try:
    # Próbáljuk betölteni a K-means centroidokat
    kmeans_centroids = pd.read_csv('ábrák/kmeans_centroids.csv', index_col=0)
    kmeans_k = len(kmeans_centroids)
    
    # K-means metrikák betöltése (ha létezik results_df mentés)
    # Mivel ez nem lett exportálva, manuálisan adjuk meg az optimális értékeket
    # MEGJEGYZÉS: Ezeket frissítsd a kmeans-v1.ipynb futtatása után!
    kmeans_silhouette = 0.4360  # Példa érték - frissítsd!
    kmeans_db = 0.9929  # Példa érték - frissítsd!
    kmeans_ch = 7045.68  # Példa érték - frissítsd!
    
    print("✓ K-means eredmények betöltve!")
    kmeans_available = True
except:
    print("⚠ K-means eredmények nem találhatók. Futtasd le először a kmeans-v1.ipynb-t!")
    print("  Folytatás becsült értékekkel...")
    kmeans_k = 3  # Becsült érték
    kmeans_silhouette = 0.4360
    kmeans_db = 0.9929
    kmeans_ch = 7045.68
    kmeans_available = False

In [ ]:
# Összehasonlító táblázat
comparison_df = pd.DataFrame({
    'Metrika': [
        'Optimális k',
        'Silhouette Score',
        'Davies-Bouldin Index',
        'Calinski-Harabasz Index'
    ],
    'K-means': [
        kmeans_k,
        kmeans_silhouette,
        kmeans_db,
        kmeans_ch
    ],
    'X-means': [
        xmeans.n_clusters_,
        silhouette,
        davies_bouldin,
        calinski_harabasz
    ],
    'Jobb érték': [
        '-',
        'Magasabb',
        'Alacsonyabb',
        'Magasabb'
    ]
})

# Nyertes meghatározása
def get_winner(row):
    if row['Metrika'] == 'Optimális k':
        return '-'
    elif row['Jobb érték'] == 'Magasabb':
        return 'K-means' if row['K-means'] > row['X-means'] else 'X-means'
    else:  # Alacsonyabb
        return 'K-means' if row['K-means'] < row['X-means'] else 'X-means'

comparison_df['Nyertes'] = comparison_df.apply(get_winner, axis=1)

print("\n" + "="*100)
print("K-MEANS vs X-MEANS ÖSSZEHASONLÍTÁS")
print("="*100)
print(comparison_df.to_string(index=False))
print("="*100)

# Exportálás
comparison_df.to_csv('ábrák/kmeans_vs_xmeans_comparison.csv', index=False, encoding='utf-8-sig')
print("\n✓ Összehasonlítás exportálva: ábrák/kmeans_vs_xmeans_comparison.csv")

In [ ]:
# Összehasonlító vizualizáció
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['Silhouette Score', 'Davies-Bouldin Index', 'Calinski-Harabasz Index']
colors_bar = ['#2ecc71', '#e74c3c']

for idx, metric in enumerate(metrics):
    ax = axes[idx]
    
    kmeans_val = comparison_df[comparison_df['Metrika'] == metric]['K-means'].values[0]
    xmeans_val = comparison_df[comparison_df['Metrika'] == metric]['X-means'].values[0]
    
    bars = ax.bar(['K-means', 'X-means'], [kmeans_val, xmeans_val], 
                  color=colors_bar, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    # Értékek annotálása
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.4f}' if metric == 'Silhouette Score' or metric == 'Davies-Bouldin Index' else f'{height:.2f}',
               ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.set_ylabel(metric, fontsize=11, fontweight='bold')
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('K-means vs X-means - Metrikák összehasonlítása', 
            fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ábrák/kmeans_vs_xmeans_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Összehasonlító diagram mentve: ábrák/kmeans_vs_xmeans_metrics.png")

## 11. Összefoglaló és Következtetések

In [ ]:
print("\n" + "="*100)
print("X-MEANS KLASZTEREZÉS - VÉGSŐ ÖSSZEFOGLALÓ")
print("="*100)

print(f"\n1. MÓDSZERTAN:")
print(f"   - Algoritmus: X-means (BIC-alapú optimalizálás)")
print(f"   - Vizsgált klaszterszámok: kmin={xmeans.kmin}, kmax={xmeans.kmax}")
print(f"   - Inicializálás: K-means++")
print(f"   - Random state: {xmeans.random_state}")
print(f"   - Használt változók: age, balance")
print(f"   - Normalizálás: StandardScaler")

print(f"\n2. X-MEANS EREDMÉNYEK:")
print(f"   - Automatikusan választott k: {xmeans.n_clusters_}")
print(f"   - BIC érték: {optimal_bic:,.2f}")
print(f"   - Silhouette Score: {silhouette:.4f}")
print(f"   - Davies-Bouldin Index: {davies_bouldin:.4f}")
print(f"   - Calinski-Harabasz Index: {calinski_harabasz:.2f}")

print(f"\n3. KLASZTEREK MÉRETE:")
for cluster_id in range(xmeans.n_clusters_):
    count = (df['cluster'] == cluster_id).sum()
    percentage = count / len(df) * 100
    avg_age = df[df['cluster'] == cluster_id]['age'].mean()
    avg_balance = df[df['cluster'] == cluster_id]['balance'].mean()
    print(f"   Klaszter {cluster_id}: {count:>6,} ügyfél ({percentage:>5.1f}%) - "
          f"Átlag kor: {avg_age:.0f} év, Átlag egyenleg: {avg_balance:,.0f} €")

print(f"\n4. K-MEANS VS X-MEANS:")
print(f"   - K-means optimális k: {kmeans_k}")
print(f"   - X-means automatikus k: {xmeans.n_clusters_}")
if xmeans.n_clusters_ != kmeans_k:
    print(f"   ⚠ Az X-means más klaszterszámot választott!")
else:
    print(f"   ✓ Mindkét módszer ugyanazt a k értéket választotta.")

print(f"\n5. EXPORTÁLT FÁJLOK:")
print(f"   CSV fájlok:")
print(f"   - ábrák/metrics_xmeans.csv - Validációs metrikák")
print(f"   - ábrák/profil_xmeans.csv - Klaszter profilok")
print(f"   - ábrák/centroids_xmeans.csv - Centroidok koordinátái")
print(f"   - ábrák/kmeans_vs_xmeans_comparison.csv - Összehasonlítás")
print(f"   \n   PNG ábrák:")
print(f"   - ábrák/xmeans_bic_plot.png - BIC értékek")
print(f"   - ábrák/xmeans_scatter_plot.png - Scatter plot")
print(f"   - ábrák/kmeans_vs_xmeans_metrics.png - Összehasonlító diagram")

print(f"\n6. KÖVETKEZTETÉSEK:")
print(f"   Az X-means algoritmus automatikusan {xmeans.n_clusters_} klasztert azonosított")
print(f"   a BIC (Bayesian Information Criterion) alapján.")
print(f"   ")
print(f"   Az X-means előnye: nincs szükség manuális k meghatározásra,")
print(f"   az algoritmus automatikusan megtalálja az optimális klaszterszámot.")

print("\n" + "="*100)
print("✓ X-MEANS KLASZTEREZÉS SIKERESEN BEFEJEZVE!")
print("="*100)